# logit-thermo A2-9：文章の結合エネルギー（GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは 10〜20 分。終わったら自動で切れます。
- ゲート：`docs/A2_9_GATE.md`。型を決めて作った短い文を Pythia-1B・OLMo-2-1B・Qwen2.5-1.5B に読ませ、名前を入れ替えて「結合」を切ったときに、
  代名詞や 2 回目の名字の −log p がどれだけ上がるかを測って判定し、`A2_9/a2_9_summary.json` に書きます。**結果も文も表示しません。**
- 計算のコード：A2-2 のデータ集めのコードは Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_9_bond.py`（`67955c837d4a93c3…`）は埋め込み（分析の部分はデータを開く前に固定）。
- 保存先 `A2_9/`。途中で切れても、もう一度「すべて実行」すれば終わったモデルは飛ばします。
- **通知**：🚀・モデルごとの ✅・🎉・⚠️。名札は「logit-thermo A2-9 文章の結合エネルギー（代名詞・名前の再登場・足し算、3 モデル、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-9 文章の結合エネルギー（代名詞・名前の再登場・足し算、3 モデル、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-9 bond energy', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 20e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU のメモリが足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
import gc, math
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRC22': 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53'})
SRC22 = _s['SRC22']
SRC9 = '"""A2-9（ゲート docs/A2_9_GATE.md）：文章の結合エネルギー。GPU（L4）で読ませ、そのあと同じノートブックで分析（CPU の計算）。\nscripts/a2_2_collect.py（load_model・valid_mask）を実行したのと同じ名前空間で実行する。分析の部分はデータを開く前に固定する。\n\n結合：\n- 代名詞の結合（P）：「名前 … Later that day, she」。名前を性別の違う名前に入れ替えて切る。結合エネルギー ＝ 切ったときの\n  代名詞の −log p − 切らないときの −log p（ナット）。名前の組は 2 つの向き（女の名前＋she と男の名前＋he）で両方使う。\n- 名前の再登場の結合（R）：「Dr. 名字 … The next morning, Dr. 名字」。1 回目の名字を別の名字に入れ替えて切る。2 回目の名字の\n  トークンの −log p の和で測る。\n- 結合の長さ：あいだに入れる、人の出てこない文（つなぎの文）の数 k ＝ 0・1・2・4・8。\n- 2 本の結合の足し算（AB）：「Dr. A and Dr. B … Dr. A … Dr. B」で、A だけ切る・B だけ切る・両方切る（k ＝ 2）。\n  2 回目の A と B のトークンの −log p の和で測る。比 ＝ 両方切った増え方 ÷（A だけ ＋ B だけ）。\nエネルギーはどれも温度 1 の −log p（使わない番号と EOS を外す。A2-8 の U と同じ）。\n"""\nimport math\nimport numpy as np\n\nA9_MODELS = [(\'pythia-1b\', \'EleutherAI/pythia-1b\'), (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\'), (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\')]\nA9_SEED = 20261113\nA9_KS = [0, 1, 2, 4, 8]\nA9_K_AB = 2\nA9_N_ITEMS = 60\nA9_BOOT, A9_BOOT_SEED = 1000, 20261114\nA9_Q1_BINS = [(0.4, \'かなり弱まる（4 割未満）\'), (0.8, \'半分くらいに弱まる（4〜8 割）\'), (1.25, \'ほとんど弱まらない（8 割〜1.25 倍）\'),\n              (float(\'inf\'), \'強くなる（1.25 倍以上）\')]\nA9_Q2_BINS = [(0.8, \'足し算より小さい（0.8 未満）\'), (1.25, \'ほぼ足し算（0.8〜1.25）\'), (float(\'inf\'), \'足し算より大きい（1.25 以上）\')]\n\nFEMALE = [\'Mary\', \'Linda\', \'Susan\', \'Karen\', \'Nancy\', \'Sarah\', \'Emily\', \'Laura\', \'Helen\', \'Anna\', \'Rachel\', \'Julia\', \'Grace\',\n          \'Alice\', \'Emma\', \'Clara\', \'Lucy\', \'Diana\', \'Rose\', \'Ellen\']\nMALE = [\'John\', \'Robert\', \'Michael\', \'David\', \'James\', \'Thomas\', \'Daniel\', \'Paul\', \'Mark\', \'Peter\', \'George\', \'Edward\', \'Henry\',\n        \'Frank\', \'Arthur\', \'Walter\', \'Samuel\', \'Oliver\', \'Victor\', \'Simon\']\nSURNAMES = [\'Thornbury\', \'Pemberton\', \'Ashcombe\', \'Whitlock\', \'Calloway\', \'Fairbairn\', \'Hollinger\', \'Marchetti\', \'Okonkwo\',\n            \'Lindqvist\', \'Bramwell\', \'Castellan\', \'Dunmore\', \'Everleigh\', \'Galbraith\', \'Halvorsen\', \'Ingersoll\', \'Kettering\',\n            \'Lockhart\', \'Montague\', \'Northcott\', \'Pellegrini\', \'Quennell\', \'Radcliffe\', \'Sinclair\', \'Tremaine\', \'Underhill\',\n            \'Vandermeer\', \'Wetherby\', \'Yarborough\', \'Abernathy\', \'Blackwood\', \'Carrington\', \'Delacroix\', \'Ellsworth\',\n            \'Fitzwilliam\', \'Gainsford\', \'Hargreaves\', \'Ivanova\', \'Jessop\']\nFILLERS = [\'The river ran slowly past the old stone bridge.\', \'Rain had fallen on the hills for most of the week.\',\n           \'A few boats were tied up along the narrow harbour wall.\', \'The market square was quiet in the early afternoon.\',\n           \'Snow still covered the higher fields above the valley.\', \'The road to the coast was lined with tall poplar trees.\',\n           \'An old clock tower stood at the end of the main street.\', \'The library had recently been painted a pale green.\',\n           \'Wind moved through the long grass beside the railway line.\', \'The bakery window was full of bread and small cakes.\',\n           \'A thin mist hung over the lake until almost noon.\', \'The museum was closed for repairs to its glass roof.\',\n           \'Several trucks waited outside the warehouse near the docks.\', \'The garden behind the school was full of roses.\',\n           \'Bells from the church could be heard across the town.\', \'The price of coffee had risen again that month.\',\n           \'A new bus route now linked the village to the city.\', \'The bridge over the canal was being widened.\',\n           \'Leaves had begun to turn red along the park paths.\', \'The station clock was running about five minutes slow.\',\n           \'Fresh fish were sold from a stall near the pier.\', \'The castle on the hill was open to visitors in summer.\',\n           \'Heavy clouds gathered over the sea in the evening.\', \'The old mill had been turned into a small hotel.\',\n           \'A long freight train passed through without stopping.\', \'The fountain in the square had been switched off.\',\n           \'Street lamps came on one by one as it grew dark.\', \'The ferry crossed the bay twice a day in winter.\',\n           \'A row of bicycles leaned against the town hall railings.\', \'The forest beyond the farms was dense and dark.\']\nVP1 = [\'opened the small bakery on the corner.\', \'arrived at the station just before nine.\', \'signed the contract at the office.\',\n       \'walked along the beach after breakfast.\', \'finished painting the kitchen walls.\', \'repaired the broken fence by the gate.\',\n       \'bought a ticket for the evening concert.\', \'planted tomatoes in the back garden.\', \'wrote a long letter to the council.\',\n       \'cleaned the windows of the old shop.\']\nVP2 = [\'walked to the market.\', \'checked the morning post.\', \'called the bank.\', \'made a pot of tea.\', \'read the newspaper.\']\n\n\ndef _fillers(rng, k):\n    return \' \'.join(rng.choice(FILLERS, size=k, replace=False).tolist()) if k else \'\'\n\n\ndef _join(*parts):\n    return \' \'.join(p for p in parts if p)\n\n\ndef build_items():\n    """→ dict(P=[...], R=[...], AB=[...])。各項目は {\'texts\': {条件: 文}, \'targets\': {条件: [(始まりの文字, 終わりの文字), ...]}}。\n    同じ項目の中では、入れ替える名前以外の文字はすべて同じ。"""\n    rng = np.random.default_rng(A9_SEED)\n    P, R, AB = [], [], []\n    for i in range(A9_N_ITEMS):\n        f, m = FEMALE[i % 20], MALE[(i * 7 + 3) % 20]\n        vp = VP1[i % len(VP1)]\n        for k in A9_KS:\n            fill = _fillers(rng, k)\n            for name, pro, other in ((f, \'she\', m), (m, \'he\', f)):\n                d = dict(item=i, k=k, pro=pro, texts={}, targets={})\n                for cond, nm in ((\'intact\', name), (\'cut\', other)):\n                    head = _join(f\'{nm} {vp}\', fill, \'Later that day,\')\n                    t = head + \' \' + pro\n                    d[\'texts\'][cond] = t; d[\'targets\'][cond] = [(len(head), len(t))]\n                P.append(d)\n        s = rng.choice(len(SURNAMES), size=4, replace=False)\n        sa, sb, sc, sd = (SURNAMES[j] for j in s)\n        for k in A9_KS:\n            fill = _fillers(rng, k)\n            d = dict(item=i, k=k, texts={}, targets={})\n            for cond, nm in ((\'intact\', sa), (\'cut\', sc)):\n                head = _join(f\'Dr. {nm} {vp}\', fill, \'The next morning, Dr.\')\n                t = head + \' \' + sa\n                d[\'texts\'][cond] = t; d[\'targets\'][cond] = [(len(head), len(t))]\n            R.append(d)\n        fill = _fillers(rng, A9_K_AB)\n        vp2 = VP2[i % len(VP2)]\n        d = dict(item=i, k=A9_K_AB, texts={}, targets={})\n        for cond, (x, y) in ((\'intact\', (sa, sb)), (\'cutA\', (sc, sb)), (\'cutB\', (sa, sd)), (\'cutAB\', (sc, sd))):\n            h1 = _join(f\'Dr. {x} and Dr. {y} met at the station.\', fill, \'The next morning, Dr.\')\n            t1 = h1 + \' \' + sa\n            h2 = t1 + \' \' + vp2 + \' After lunch, Dr.\'\n            t = h2 + \' \' + sb\n            d[\'texts\'][cond] = t; d[\'targets\'][cond] = [(len(h1), len(t1)), (len(h2), len(t))]\n        AB.append(d)\n    return dict(P=P, R=R, AB=AB)\n\n\n# ---------- 読ませる（GPU） ----------\n\ndef target_tokens(tok, text, spans):\n    """→ (トークン列, 的のトークンの位置のリスト)。的の文字の区間が、トークンの切れ目にぴったり合うことを確かめる。"""\n    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True)\n    ids, off = list(enc[\'input_ids\']), list(enc[\'offset_mapping\'])\n    bos = [tok.bos_token_id] if (tok.bos_token_id is not None and getattr(tok, \'add_bos_token\', False)) else []\n    pos = []\n    for a, b in spans:\n        idx = [j for j, (s, e) in enumerate(off) if s < b and e > a]\n        assert idx, (\'的のトークンがありません\', text[a:b])\n        # 区間の頭がトークンの頭と合う（前の語とくっついていない）。空白込みのトークンは、頭が a か a+1\n        assert off[idx[0]][0] in (a, a + 1) and off[idx[-1]][1] == b, (\'的の区間がトークンの切れ目と合いません\', text[a:b], [off[j] for j in idx])\n        pos += [j + len(bos) for j in idx]\n    return bos + ids, pos\n\n\ndef score_texts(tok, model, valid, texts_spans, device, bs=8):\n    """texts_spans：[(文, 区間のリスト)]。→ 各文の的のトークンの −log p₁ の和（ナット、float64）と、的のトークンの数。"""\n    import torch\n    enc = [target_tokens(tok, t, sp) for t, sp in texts_spans]\n    out, ntok = np.zeros(len(enc)), np.zeros(len(enc), int)\n    pad = tok.pad_token_id if tok.pad_token_id is not None else 0\n    with torch.no_grad():\n        for i0 in range(0, len(enc), bs):\n            chunk = enc[i0:i0 + bs]\n            Lm = max(len(ids) for ids, _ in chunk)\n            x = torch.full((len(chunk), Lm), pad, dtype=torch.long, device=device)\n            att = torch.zeros((len(chunk), Lm), dtype=torch.long, device=device)\n            for b, (ids, _) in enumerate(chunk):\n                x[b, :len(ids)] = torch.tensor(ids, device=device); att[b, :len(ids)] = 1\n            lg = model(input_ids=x, attention_mask=att).logits.double()\n            lg = lg.masked_fill(~valid[None, None, :], float(\'-inf\'))\n            lp = torch.log_softmax(lg, dim=-1)\n            for b, (ids, pos) in enumerate(chunk):\n                p = torch.tensor(pos, device=device)\n                y = torch.tensor([ids[j] for j in pos], device=device)\n                out[i0 + b] = float(-lp[b, p - 1, y].sum())\n                ntok[i0 + b] = len(pos)\n    return out, ntok\n\n\ndef run_bond(name, hf_id, items, device):\n    tok, model = load_model(hf_id, device)\n    valid, _ = valid_mask(tok, model, device)\n    res = {}\n    for kind, lst in items.items():\n        conds = list(lst[0][\'texts\'])\n        flat = [(d[\'texts\'][c], d[\'targets\'][c]) for d in lst for c in conds]\n        e, n = score_texts(tok, model, valid, flat, device)\n        res[kind] = dict(conds=conds, E=e.reshape(len(lst), len(conds)).tolist(), ntok=n.reshape(len(lst), len(conds)).tolist())\n    del model\n    return res\n\n\ndef self_test_score9(device):\n    """小さなランダムのモデルで：まとめて読ませた値と、1 本ずつ・詰め物なしで読ませた値が合う。"""\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    valid, _ = valid_mask(tok, model, device)\n    it = build_items()\n    ts = [(d[\'texts\'][c], d[\'targets\'][c]) for d in it[\'P\'][:6] + it[\'AB\'][:3] for c in d[\'texts\']]\n    a, _ = score_texts(tok, model, valid, ts, device, bs=8)\n    b, _ = score_texts(tok, model, valid, ts, device, bs=1)\n    assert np.abs(a - b).max() < 1e-3, np.abs(a - b).max()\n    return \'OK\'\n\n\n# ---------- 分析（CPU） ----------\n\ndef bond_energies(res):\n    """→ P・R：(項目, k) の結合エネルギー（P は 2 つの向きの平均）、AB：(項目, 3) の [A だけ, B だけ, 両方] の増え方。"""\n    out = {}\n    for kind in (\'P\', \'R\'):\n        r = res[kind]\n        E = np.array(r[\'E\'])\n        ci, cc = r[\'conds\'].index(\'intact\'), r[\'conds\'].index(\'cut\')\n        dE = E[:, cc] - E[:, ci]\n        per = 2 if kind == \'P\' else 1\n        out[kind] = dE.reshape(A9_N_ITEMS, len(A9_KS), per).mean(-1)\n    r = res[\'AB\']\n    E = np.array(r[\'E\'])\n    c = {k: r[\'conds\'].index(k) for k in r[\'conds\']}\n    out[\'AB\'] = np.stack([E[:, c[\'cutA\']] - E[:, c[\'intact\']], E[:, c[\'cutB\']] - E[:, c[\'intact\']], E[:, c[\'cutAB\']] - E[:, c[\'intact\']]], 1)\n    return out\n\n\ndef stats_model(be, rng):\n    k0, k8 = A9_KS.index(0), A9_KS.index(8)\n    f_ret = lambda X: float(X[:, k8].mean() / X[:, k0].mean()) if X[:, k0].mean() > 0 else float(\'nan\')\n    f_add = lambda X: float(X[:, 2].mean() / (X[:, 0].mean() + X[:, 1].mean())) if (X[:, 0].mean() + X[:, 1].mean()) > 0 else float(\'nan\')\n    pt = dict(P_by_k=be[\'P\'].mean(0).tolist(), R_by_k=be[\'R\'].mean(0).tolist(), AB=be[\'AB\'].mean(0).tolist(),\n              P_ret=f_ret(be[\'P\']), R_ret=f_ret(be[\'R\']), add=f_add(be[\'AB\']))\n    bs = {k: [] for k in (\'P0\', \'P_ret\', \'R_ret\', \'add\')}\n    for _ in range(A9_BOOT):\n        ii = rng.integers(0, A9_N_ITEMS, A9_N_ITEMS)\n        bs[\'P0\'].append(float(be[\'P\'][ii, 0].mean())); bs[\'P_ret\'].append(f_ret(be[\'P\'][ii]))\n        bs[\'R_ret\'].append(f_ret(be[\'R\'][ii])); bs[\'add\'].append(f_add(be[\'AB\'][ii]))\n    ci = {k: [float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))] for k, v in bs.items()}\n    pt[\'ci\'] = ci\n    pt[\'P_bond_exists\'] = bool(ci[\'P0\'][0] > 0)\n    return pt\n\n\ndef bin_word(x, bins):\n    if not np.isfinite(x):\n        return \'はっきりしない\'\n    if x < 0:\n        return \'向きが逆\'\n    for edge, w in bins:\n        if x < edge:\n            return w\n\n\ndef analyze9(allres):\n    rng = np.random.default_rng(A9_BOOT_SEED)\n    out = {\'models\': {}}\n    for name, _ in A9_MODELS:\n        be = bond_energies(allres[name])\n        out[\'models\'][name] = stats_model(be, rng)\n    ms = [out[\'models\'][m] for m, _ in A9_MODELS]\n    rets = [m[\'P_ret\'] for m in ms if m[\'P_bond_exists\'] and np.isfinite(m[\'P_ret\'])]\n    out[\'Q1\'] = bin_word(float(np.median(rets)), A9_Q1_BINS) if len(rets) >= 2 else \'はっきりしない\'\n    adds = [m[\'add\'] for m in ms if np.isfinite(m[\'add\'])]\n    out[\'Q2\'] = bin_word(float(np.median(adds)), A9_Q2_BINS) if len(adds) >= 2 else \'はっきりしない\'\n    out[\'Q1_each\'], out[\'Q2_each\'] = rets, adds\n    return out\n\n\ndef self_test_a9():\n    it = build_items()\n    assert len(it[\'P\']) == A9_N_ITEMS * len(A9_KS) * 2 and len(it[\'R\']) == A9_N_ITEMS * len(A9_KS) and len(it[\'AB\']) == A9_N_ITEMS\n    for d in it[\'P\'][:40] + it[\'R\'][:40] + it[\'AB\'][:20]:\n        ts = list(d[\'texts\'].values())\n        assert all(len(t) > 0 for t in ts) and len(set(ts)) == len(ts)\n        for c, t in d[\'texts\'].items():\n            for a, b in d[\'targets\'][c]:\n                assert t[a] == \' \' and t[a + 1:b].strip() == t[a + 1:b]\n        # 的の中身はどの条件でも同じ\n        tg = [[d[\'texts\'][c][a:b] for a, b in d[\'targets\'][c]] for c in d[\'texts\']]\n        assert all(x == tg[0] for x in tg)\n    # 人の出てこないつなぎの文\n    for s in FILLERS:\n        w = set(s.replace(\'.\', \'\').split())\n        assert not (w & {\'he\', \'she\', \'He\', \'She\', \'his\', \'her\', \'His\', \'Her\', \'him\', \'man\', \'woman\'}) and not (w & set(FEMALE + MALE + SURNAMES)), s\n    # 作り物の分析：P は距離で半分に、R は変わらず、AB はちょうど足し算\n    fake = {}\n    rng = np.random.default_rng(0)\n    for name, _ in A9_MODELS:\n        nP, nR = A9_N_ITEMS * len(A9_KS) * 2, A9_N_ITEMS * len(A9_KS)\n        kk = np.tile(np.repeat(np.array(A9_KS), 2), A9_N_ITEMS)\n        EP = np.stack([np.full(nP, 3.0), 3.0 + 2.0 * np.exp(-kk / 8 * math.log(2)) + rng.normal(0, 0.2, nP)], 1)\n        ER = np.stack([np.full(nR, 1.0), 9.0 + rng.normal(0, 0.2, nR)], 1)\n        EA = np.stack([np.zeros(A9_N_ITEMS), 5 + rng.normal(0, .1, A9_N_ITEMS), 6 + rng.normal(0, .1, A9_N_ITEMS), 11 + rng.normal(0, .1, A9_N_ITEMS)], 1)\n        fake[name] = dict(P=dict(conds=[\'intact\', \'cut\'], E=EP.tolist()), R=dict(conds=[\'intact\', \'cut\'], E=ER.tolist()),\n                          AB=dict(conds=[\'intact\', \'cutA\', \'cutB\', \'cutAB\'], E=EA.tolist()))\n    r = analyze9(fake)\n    m = r[\'models\'][\'pythia-1b\']\n    assert abs(m[\'P_ret\'] - 0.5) < 0.05 and abs(m[\'R_ret\'] - 1) < 0.05 and abs(m[\'add\'] - 1) < 0.03, m\n    assert r[\'Q1\'] == \'半分くらいに弱まる（4〜8 割）\' and r[\'Q2\'] == \'ほぼ足し算（0.8〜1.25）\', r\n    return \'OK\'\n'
assert hashlib.sha256(SRC9.encode()).hexdigest() == '67955c837d4a93c3c4f60181c896c4222821935e210529673ad53abf618e46ef'
exec(SRC22)
exec(SRC9)
def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.floating, np.integer)): o = o.item()
    if isinstance(o, float) and not math.isfinite(o): return None
    return o
print('コードを置きました')


In [ ]:
# セル3：自己テスト → 読ませる（モデルごと）→ 分析 → 保存（結果は表示しない）
import json, sys, transformers
try:
    print('自己テスト：', self_test_a9(), self_test_score9('cuda'))
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_9', exist_ok=True)
    items = build_items()
    allres = {}
    for name, hf_id in A9_MODELS:
        p = '/content/drive/MyDrive/logit-thermo/A2_9' + f'/raw_{name}.json'
        if not os.path.exists(p):
            t0 = time.time()
            r = run_bond(name, hf_id, items, 'cuda'); gc.collect(); torch.cuda.empty_cache()
            json.dump(r, open(p + '.part', 'w')); os.replace(p + '.part', p)
            notify(f'✅ {name} おわり（{time.time() - t0:.0f} 秒）', tags='white_check_mark')
        allres[name] = json.load(open(p))
    r = analyze9(allres)
    r.update(gate='docs/A2_9_GATE.md', a2_9_sha256='67955c837d4a93c3c4f60181c896c4222821935e210529673ad53abf618e46ef', a2_2_collect_sha256='e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53',
             torch=torch.__version__, transformers=transformers.__version__, python=sys.version.split()[0],
             gpu=torch.cuda.get_device_name(0), matmul=torch.get_float32_matmul_precision())
    json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo/A2_9' + '/a2_9_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-9（文章の結合エネルギー）で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：A2-9 の判定を A2_9/a2_9_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
